# Onde a riva — Definizione dei siti di studio

Un sito alla volta: scegli il punto costiero e quello offshore, carica i profili QGIS, verifica la pendenza della battigia, prepara ERA5 ed esporta un pacchetto per il portale.

**Il notebook è autosufficiente:** le funzioni del progetto sono incluse nella cella tecnica; non occorre clonare GitHub o pubblicare il sito. Parti in modalità dimostrativa per vedere tutti i passaggi senza credenziali. Per i tuoi dati disattiva `DEMO` e usa le celle in ordine.

La pendenza segue **Stockdon_runup_soluzione.ipynb**, celle 24–25: orientamento mare→terra dalla media delle prime/ultime 10 quote; distanza euclidea X/Y dal primo punto a mare; selezione prima del primo punto sopra `Z_MAX`; regressione con `np.polyfit`. Le formule di runup del portale restano invariate.

Il pacchetto finale non contiene token o credenziali. Il notebook non pubblica e non modifica automaticamente la repository.

## 1. Ambiente e cartella di lavoro
Apri in Colab con **File → Carica notebook**. È sufficiente il runtime CPU. L’installazione riguarda soltanto questo runtime.

In [ ]:
#@title Installa le librerie
INSTALLA_DIPENDENZE = True #@param {type:"boolean"}
if INSTALLA_DIPENDENZE:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "numpy>=1.24", "pandas>=2", "matplotlib>=3.7", "folium>=0.14", "pyproj>=3.6", "cdsapi>=0.7.7,<1", "netCDF4>=1.7,<2"])
print("Ambiente pronto.")

In [ ]:
#@title Carica le funzioni del progetto (cella tecnica)
# Codice condiviso con la repository, incluso per funzionare anche prima del push.
from pathlib import Path
import json, sys, tempfile, hashlib
RUNTIME = Path(tempfile.mkdtemp(prefix="ondeariva_runtime_"))
BUNDLED_SOURCES = {'forecast_service/__init__.py': '"""Calcolo e pubblicazione del portale locale, indipendente da Streamlit."""\n', 'forecast_service/site_setup.py': '"""Beach-face slope following Stockdon_runup_soluzione.ipynb, cells 24–25.\n\nThe supplied teaching notebook defines selection and orientation. Diagnostics\nreject degenerate fits; a negative slope is not converted to an absolute value.\n"""\nimport math\n\nBEACHFACE_METHOD_VERSION = \'stockdon_teaching_notebook_v1\'\n\n\ndef prepare_profile(frame, *, distance_col=\'distance\', elevation_col=\'SAMPLE_1\',\n                    x_col=\'X\', y_col=\'Y\', orientation=\'auto\'):\n    import numpy as np\n    import pandas as pd\n    if orientation not in (\'auto\',\'distance_increases_landward\',\'distance_increases_seaward\'):\n        raise ValueError(\'Orientamento non riconosciuto.\')\n    spatial = x_col is not None and y_col is not None\n    if (x_col is None) != (y_col is None):\n        raise ValueError(\'Specificare entrambe le colonne X/Y oppure nessuna.\')\n    required=[distance_col,elevation_col]+([x_col,y_col] if spatial else [])\n    missing=set(required)-set(frame.columns)\n    if missing:\n        raise ValueError(f\'Colonne mancanti: {sorted(missing)}\')\n    profile=frame.copy()\n    for name in required:\n        profile[name]=pd.to_numeric(profile[name],errors=\'raise\')\n    dropped=int(profile[elevation_col].isna().sum())\n    profile=profile.dropna(subset=[elevation_col]).sort_values(distance_col)\n    if len(profile)<2 or not np.isfinite(profile[required].to_numpy(dtype=float)).all():\n        raise ValueError(\'Servono almeno due punti con distanze, coordinate e quote finite.\')\n    if profile[distance_col].duplicated().any():\n        raise ValueError(\'Distanze duplicate nel transetto: verificare il CSV.\')\n    sea_at_end=bool(profile[elevation_col].iloc[:10].mean()>profile[elevation_col].iloc[-10:].mean())\n    ambiguous=bool(abs(profile[elevation_col].iloc[:10].mean()-profile[elevation_col].iloc[-10:].mean())<1e-12)\n    flip=sea_at_end if orientation==\'auto\' else orientation==\'distance_increases_seaward\'\n    if flip:\n        profile=profile.iloc[::-1]\n    profile=profile.reset_index(drop=True)\n    if spatial:\n        profile[\'x_mare\']=np.hypot(profile[x_col]-profile[x_col].iloc[0],profile[y_col]-profile[y_col].iloc[0])\n    else:\n        # Compatibility with 1-D profiles: declared separately in the metadata.\n        profile[\'x_mare\']=(profile[distance_col]-profile[distance_col].iloc[0]).abs()\n    profile.attrs.update(reversed=bool(flip),orientation=orientation,orientation_ambiguous=ambiguous,\n                         dropped_nan_z=dropped,distance_source=\'euclidean_xy\' if spatial else \'one_dimensional_distance\')\n    return profile\n\n\ndef fit_beachface(profile,z_min,z_max,*,elevation_col=\'SAMPLE_1\'):\n    import numpy as np\n    if not math.isfinite(z_min) or not math.isfinite(z_max) or z_min>=z_max:\n        raise ValueError(\'Definire un intervallo di quote crescente.\')\n    # Same sea-connected prefix and regression as pendenza_battigia in cell 25.\n    above=np.where(profile[elevation_col]>z_max)[0]\n    stop=int(above[0]) if len(above) else len(profile)\n    selected=(profile[elevation_col]>=z_min)&(profile.index<stop)\n    x=profile.loc[selected,\'x_mare\'].to_numpy(dtype=float)\n    z=profile.loc[selected,elevation_col].to_numpy(dtype=float)\n    if len(x)<2 or len(np.unique(x))<2:\n        raise ValueError(\'Meno di due distanze distinte prima del primo punto sopra Z_MAX.\')\n    slope,intercept=np.polyfit(x,z,1)\n    if not math.isfinite(slope) or not 0<slope<1:\n        raise ValueError(f\'Pendenza {slope:.6g} non valida: controllare orientamento e tratto di battigia.\')\n    variance=float(np.sum((z-z.mean())**2))\n    r2=1-float(np.sum((z-(slope*x+intercept))**2))/variance if variance>0 else 0.0\n    diagnostics={\'slope\':float(slope),\'n_points\':int(len(x)),\'r_squared\':r2,\'fit_intercept_m\':float(intercept),\n                 \'z_min_m\':float(z_min),\'z_max_m\':float(z_max),\'stop_index\':stop,\n                 \'first_above_zmax_found\':bool(len(above)),\'selected_x_min_m\':float(x.min()),\n                 \'selected_x_max_m\':float(x.max()),\'method\':BEACHFACE_METHOD_VERSION,**profile.attrs}\n    return diagnostics,selected\n\n\ndef analyze_transects(frame,z_min,z_max,*,id_col=\'TR_ID\',distance_col=\'distance\',elevation_col=\'SAMPLE_1\',\n                      x_col=\'X\',y_col=\'Y\',orientation=\'auto\'):\n    import pandas as pd\n    if id_col not in frame or frame[id_col].isna().any():\n        raise ValueError(\'Colonna identificativo assente o con valori mancanti.\')\n    summaries=[]; profiles={}; selections={}\n    for identifier,group in frame.groupby(id_col,sort=False):\n        identifier=str(identifier)\n        try:\n            p=prepare_profile(group,distance_col=distance_col,elevation_col=elevation_col,\n                              x_col=x_col,y_col=y_col,orientation=orientation)\n            summary,selected=fit_beachface(p,z_min,z_max,elevation_col=elevation_col)\n            summaries.append(dict(summary,transect_id=identifier,status=\'valid\',error=\'\'))\n            profiles[identifier]=p; selections[identifier]=selected\n        except (ValueError,KeyError,TypeError) as error:\n            summaries.append({\'transect_id\':identifier,\'status\':\'invalid\',\'error\':str(error),\'slope\':None})\n    return pd.DataFrame(summaries),profiles,selections\n\n\ndef slopes_from_transects(path,z_min,z_max):\n    import pandas as pd\n    frame=pd.read_csv(path)\n    if {\'TR_ID\',\'distance\',\'SAMPLE_1\',\'X\',\'Y\'}<=set(frame):\n        options={}\n    elif {\'transect_id\',\'x_m\',\'z_m\'}<=set(frame):\n        options=dict(id_col=\'transect_id\',distance_col=\'x_m\',elevation_col=\'z_m\',x_col=None,y_col=None)\n    else:\n        raise ValueError(\'CSV QGIS: TR_ID,distance,SAMPLE_1,X,Y; oppure CSV 1-D: transect_id,x_m,z_m.\')\n    summary,_,_=analyze_transects(frame,z_min,z_max,**options)\n    if summary.empty:\n        raise ValueError(\'Il CSV non contiene transetti.\')\n    invalid=summary[summary.status!=\'valid\']\n    if not invalid.empty:\n        raise ValueError(\'; \'.join(f"{r.transect_id}: {r.error}" for r in invalid.itertuples()))\n    return summary.drop(columns=[\'status\',\'error\']).to_dict(\'records\')\n', 'forecast_service/time_utils.py': '"""Pure helpers shared with the standalone Colab setup."""\nimport math\nfrom datetime import datetime,timezone\n\n\ndef iso(dt):\n    return dt.astimezone(timezone.utc).isoformat().replace(\'+00:00\',\'Z\')\n\n\ndef parse_time(value):\n    result=value if isinstance(value,datetime) else datetime.fromisoformat(str(value).replace(\'Z\',\'+00:00\'))\n    return result.replace(tzinfo=timezone.utc) if result.tzinfo is None else result.astimezone(timezone.utc)\n\n\ndef quantile(values,q):\n    ordered=sorted(values)\n    position=(len(ordered)-1)*q\n    low,high=math.floor(position),math.ceil(position)\n    return ordered[low]+(ordered[high]-ordered[low])*(position-low)\n', 'forecast_service/era5_download.py': '"""ERA5 adapter based on wave_climate/download.py, Alessio Rovere (MIT, 2026).\n\nKeep request caching, NetCDF/ZIP handling, nearest sea cell and adaptive chunks.\nOnly waves are needed here. Mean period is never renamed to peak period.\nSee LICENSE-wave-climate.txt for the upstream license.\n"""\nimport datetime as dt\nimport hashlib\nimport io\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\nimport zipfile\n\nERA5_TS = \'reanalysis-era5-single-levels-timeseries\'\nVARIABLES = [\'significant_height_of_combined_wind_waves_and_swell\', \'mean_wave_period\', \'mean_wave_direction\']\nSHORT = {\'swh\':\'hs_m\', \'mwp\':\'tm_s\', \'mwd\':\'direction_deg\',\n         \'significant_height_of_combined_wind_waves_and_swell\':\'hs_m\',\n         \'mean_wave_period\':\'tm_s\', \'mean_wave_direction\':\'direction_deg\'}\n\n\nclass TooLarge(Exception):\n    pass\n\n\ndef cds_client():\n    import cdsapi\n    key = os.environ.get(\'CDS_KEY\')\n    if key:\n        return cdsapi.Client(url=os.environ.get(\'CDS_URL\',\'https://cds.climate.copernicus.eu/api\'),key=key,quiet=True)\n    return cdsapi.Client(quiet=True)  # Also supports the user\'s standard ~/.cdsapirc.\n\n\ndef open_cds(path):\n    import xarray as xr\n    if zipfile.is_zipfile(path):\n        frames = []\n        with zipfile.ZipFile(path) as archive:\n            for name in archive.namelist():\n                if name.endswith(\'.nc\'):\n                    with xr.open_dataset(io.BytesIO(archive.read(name))) as ds:\n                        frames.append(ds.load())\n        if not frames:\n            raise ValueError(\'Archivio CDS senza file NetCDF.\')\n        return xr.merge(frames,compat=\'override\')\n    with xr.open_dataset(path) as ds:\n        return ds.load()\n\n\ndef ts_frame(ds):\n    import pandas as pd\n    df=ds.to_dataframe().reset_index()\n    tcol=\'valid_time\' if \'valid_time\' in df else \'time\'\n    keep={v:SHORT[v] for v in ds.data_vars if v in SHORT}\n    if \'hs_m\' not in keep.values():\n        raise ValueError(\'La risposta CDS non contiene Hs.\')\n    out=df[[tcol,*keep,*[c for c in (\'latitude\',\'longitude\') if c in df]]].rename(columns={tcol:\'time\',**keep})\n    out[\'time\']=pd.to_datetime(out[\'time\'],utc=True)\n    return out\n\n\ndef cached_fetch(client, dataset, request, cache, retries=3):\n    cache=Path(cache); cache.mkdir(parents=True,exist_ok=True)\n    key=hashlib.sha256(json.dumps({\'dataset\':dataset,**request},sort_keys=True).encode()).hexdigest()[:16]\n    path=cache/f\'era5_{key}.nc\'\n    if path.exists():\n        return path\n    partial=path.with_suffix(\'.part\')\n    for attempt in range(retries):\n        try:\n            client.retrieve(dataset,request,str(partial))\n            ds=open_cds(partial)\n            try:\n                ts_frame(ds)\n            finally:\n                ds.close()\n            partial.replace(path)\n            return path\n        except Exception as error:\n            partial.unlink(missing_ok=True)\n            message=str(error).lower()\n            if any(word in message for word in (\'too large\',\'cost limit\',\'request too big\')):\n                raise TooLarge() from error\n            if attempt==retries-1:\n                raise\n            time.sleep(30*(attempt+1))\n\n\ndef ts_range(client, lon, lat, start, end, cache, retries=3):\n    request={\'variable\':sorted(VARIABLES),\'location\':{\'longitude\':round(lon,4),\'latitude\':round(lat,4)},\n             \'date\':[f\'{start.isoformat()}/{end.isoformat()}\'],\'data_format\':\'netcdf\'}\n    try:\n        return [cached_fetch(client,ERA5_TS,request,cache,retries)]\n    except TooLarge:\n        if (end-start).days<=31:\n            raise ValueError(\'Richiesta CDS troppo grande anche dopo la suddivisione.\')\n        middle=start+dt.timedelta(days=(end-start).days//2)\n        return ts_range(client,lon,lat,start,middle,cache,retries)+ts_range(client,lon,lat,middle+dt.timedelta(days=1),end,cache,retries)\n\n\ndef nearest_sea_cell(client, lon, lat, cache, search_deg=.75):\n    # CDS time-series accepts a location. Use the gridded product for the area\n    # probe, instead of sending the upstream script\'s area to the point service.\n    request={\'product_type\':[\'reanalysis\'],\'variable\':[VARIABLES[0]],\'year\':[\'2020\'],\'month\':[\'01\'],\n             \'day\':[\'01\'],\'time\':[f\'{h:02d}:00\' for h in range(24)],\n             \'area\':[lat+search_deg,lon-search_deg,lat-search_deg,lon+search_deg],\n             \'data_format\':\'netcdf\',\'download_format\':\'unarchived\'}\n    ds=open_cds(cached_fetch(client,\'reanalysis-era5-single-levels\',request,cache))\n    try:\n        frame=ts_frame(ds)\n    finally:\n        ds.close()\n    cells=frame.groupby([\'latitude\',\'longitude\'])[\'hs_m\'].apply(lambda v:v.notna().mean()).reset_index()\n    cells=cells[cells.hs_m>.9]\n    if cells.empty:\n        raise ValueError(\'Nessuna cella marina ERA5 vicina: scegliere un punto più al largo.\')\n    delta=((cells.longitude-lon+180)%360)-180\n    distance=(delta*math.cos(math.radians(lat)))**2+(cells.latitude-lat)**2\n    cell=cells.loc[distance.idxmin()]\n    return float((cell.longitude+180)%360-180),float(cell.latitude)\n\n\ndef year_chunks(start,end,years=10):\n    if end<start or years<1:\n        raise ValueError(\'Periodo ERA5 non valido.\')\n    chunks=[]\n    current=start\n    while current<=end:\n        stop=min(dt.date(current.year+years-1,12,31),end)\n        chunks.append((current,stop))\n        current=stop+dt.timedelta(days=1)\n    return chunks\n\n\ndef download_series(location, start, end, cache, *, client=None, cell=None):\n    import pandas as pd\n    if end>dt.datetime.now(dt.timezone.utc).date()-dt.timedelta(days=6):\n        raise ValueError(\'Usare una data finale almeno sei giorni prima di oggi per ERA5.\')\n    client=client or cds_client()\n    point=location[\'wave_point\']\n    lon,lat=cell or nearest_sea_cell(client,point[\'longitude\'],point[\'latitude\'],cache)\n    frames=[]\n    for a,b in year_chunks(start,end):\n        print(f\'ERA5 {a} → {b}, cella {lat:.2f} N {lon:.2f} E (cache delle richieste attiva)\')\n        for path in ts_range(client,lon,lat,a,b,cache):\n            ds=open_cds(path)\n            try:\n                frames.append(ts_frame(ds))\n            finally:\n                ds.close()\n    frame=pd.concat(frames).drop_duplicates(\'time\',keep=\'last\').sort_values(\'time\')\n    frame=frame[(frame.time>=pd.Timestamp(start,tz=\'UTC\')) & (frame.time<pd.Timestamp(end+dt.timedelta(days=1),tz=\'UTC\'))]\n    # Reject mixed cells: a local reference must represent one known marine cell.\n    if any(c in frame and frame[c].nunique(dropna=True)>1 for c in (\'latitude\',\'longitude\')):\n        raise ValueError(\'Il CDS ha restituito più celle per la serie puntuale.\')\n    requested_count=(end-start).days*24+24\n    valid=frame[\'hs_m\'].notna()\n    info={\'dataset_id\':ERA5_TS,\'doi\':\'10.24381/1cf1ad76\',\'cell_lon\':lon,\'cell_lat\':lat,\n          \'requested_start\':str(start),\'requested_end\':str(end),\'requested_hours\':requested_count,\n          \'returned_records\':len(frame),\'valid_hs_records\':int(valid.sum()),\n          \'missing_hs_records\':requested_count-int(valid.sum()),\n          \'period_type\':\'mean_wave_period_not_peak\',\'downloaded_at\':dt.datetime.now(dt.timezone.utc).isoformat()}\n    if not valid.any():\n        raise ValueError(\'La cella ERA5 non contiene altezze d’onda valide.\')\n    return frame,info\n', 'forecast_service/climate.py': '"""Offline historical reference, reused by the two daily forecast updates."""\nfrom bisect import bisect_right\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport math\n\n\ndef build_climate(csv_path, source, *, time_column=\'time\', hs_column=\'hs_m\'):\n    from forecast_service.time_utils import iso, parse_time, quantile\n    rows = []\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        for row in csv.DictReader(handle):\n            value = float(row[hs_column])\n            if not math.isfinite(value) or value < 0:\n                raise ValueError(\'La serie storica contiene Hs mancanti o negative: pulire i dati esplicitamente.\')\n            rows.append((parse_time(row[time_column]), value))\n    rows.sort()\n    if len(rows) < 2 or len({t for t,_ in rows}) != len(rows):\n        raise ValueError(\'Servono almeno due tempi distinti, senza duplicati.\')\n    values = sorted(value for _,value in rows)\n    annual = {}\n    for time, value in rows:\n        year = str(time.year)\n        if year not in annual or value > annual[year][\'hs_m\']:\n            annual[year] = {\'hs_m\': value, \'time\': iso(time)}\n    gaps = [(b[0]-a[0]).total_seconds()/3600 for a,b in zip(rows,rows[1:])]\n    step = quantile(gaps, .5)\n    # Irregular sampling would bias an observation-count percentile.\n    if any(abs(g/step-round(g/step)) > .001 for g in gaps) or step <= 0:\n        raise ValueError(\'Campionamento non regolare: uniformare la serie prima del confronto climatologico.\')\n    return {\'schema_version\':1, \'source\':source, \'units\':\'m\', \'start\':iso(rows[0][0]), \'end\':iso(rows[-1][0]),\n            \'n_records\':len(rows), \'time_step_hours\':step,\n            \'coverage_fraction\':len(rows)/(int(round((rows[-1][0]-rows[0][0]).total_seconds()/3600/step))+1),\n            \'hs_sorted_m\':values, \'p95_m\':quantile(values,.95), \'p99_m\':quantile(values,.99),\n            \'max_hs_m\':max(values), \'annual_maxima\':annual,\n            \'input_sha256\':hashlib.sha256(Path(csv_path).read_bytes()).hexdigest(),\n            \'method\':\'Empirical CDF, all seasons and directions; annual maxima are observations, not return levels.\'}\n\n\ndef load_climate(root, relative):\n    path = (root / relative).resolve()\n    if not path.is_relative_to(root.resolve()):\n        raise ValueError(\'Il riferimento climatologico deve essere nella repository.\')\n    data = json.loads(path.read_text())\n    values = data[\'hs_sorted_m\']\n    if data[\'schema_version\'] != 1 or data[\'units\'] != \'m\' or not values or values != sorted(values) or any(not math.isfinite(v) or v < 0 for v in values):\n        raise ValueError(\'Riferimento climatologico non valido.\')\n    if len(values) != data[\'n_records\']:\n        raise ValueError(\'Conteggio della serie storica non coerente.\')\n    return data\n\n\ndef compare_waves(waves, climate):\n    if not climate:\n        return waves, {\'status\':\'not_configured\', \'description\':\'Climatologia non ancora caricata per questa località.\'}\n    values = climate[\'hs_sorted_m\']\n    rows = [dict(wave, hs_percentile=100*bisect_right(values,wave[\'hs_m\'])/len(values)) for wave in waves]\n    summary = {key:value for key,value in climate.items() if key != \'hs_sorted_m\'}\n    summary[\'status\'] = \'available\'\n    summary[\'comparison_note\'] = \'Confronto indicativo fra previsione Copernicus e serie storica; eventuali bias fra prodotti non sono corretti. Il percentile non è un periodo di ritorno.\'\n    return rows, summary\n', 'forecast_service/site_package.py': '"""Portable study-site packages exported by the standalone Colab notebook."""\nimport copy\nimport json\nfrom pathlib import Path\nimport re\nimport tempfile\nimport zipfile\n\n\ndef validate_payload(payload):\n    slug=payload.get(\'slug\',\'\')\n    if not re.fullmatch(\'[a-z0-9]+(?:-[a-z0-9]+)*\',slug):\n        raise ValueError(\'Identificativo della località non valido.\')\n    if payload.get(\'schema_version\')!=1 or payload.get(\'data_mode\') not in (\'study\',\'demo\'):\n        raise ValueError(\'Formato del pacchetto non riconosciuto.\')\n    if payload.get(\'data_mode\')!=\'demo\' and any(payload.get(\'provenance\',{}).get(key)==\'demo\' for key in (\'profile_data_mode\',\'climate_data_mode\')):\n        raise ValueError(\'Dati dimostrativi dichiarati come dati di studio.\')\n    location=payload[\'location\']\n    if not location.get(\'slopes\') or any(not isinstance(s,(int,float)) or not 0<s<1 for s in location[\'slopes\']):\n        raise ValueError(\'Pendenze non valide.\')\n    if not payload.get(\'thresholds\'):\n        raise ValueError(\'Soglie mancanti.\')\n    if location.get(\'climatology_file\') not in (None,f\'config/climatology/{slug}.json\'):\n        raise ValueError(\'Percorso climatologico non valido nel pacchetto.\')\n    json.dumps(payload,allow_nan=False)\n    return slug\n\n\ndef write_site_package(output,payload,*,climatology=None,attachments=None):\n    slug=validate_payload(payload)\n    if bool(climatology) != bool(payload[\'location\'].get(\'climatology_file\')):\n        raise ValueError(\'Riferimento climatologico e configurazione non coerenti.\')\n    if climatology:\n        from forecast_service.climate import load_climate\n        with tempfile.TemporaryDirectory() as folder:\n            root=Path(folder)\n            (root/\'climatology.json\').write_text(json.dumps(climatology,allow_nan=False))\n            load_climate(root,\'climatology.json\')\n    output=Path(output); output.parent.mkdir(parents=True,exist_ok=True)\n    temporary=output.with_suffix(\'.zip.tmp\')\n    try:\n        with zipfile.ZipFile(temporary,\'w\',compression=zipfile.ZIP_DEFLATED) as archive:\n            archive.writestr(\'site.json\',json.dumps(payload,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\')\n            if climatology:\n                archive.writestr(\'climatology.json\',json.dumps(climatology,ensure_ascii=False,allow_nan=False,separators=(\',\',\':\'))+\'\\n\')\n            for name,path in (attachments or {}).items():\n                if not re.fullmatch(\'[A-Za-z0-9_.-]+\',name) or name.startswith(\'.\'):\n                    raise ValueError(\'Nome allegato non valido.\')\n                archive.write(path,\'diagnostics/\'+name)\n        temporary.replace(output)\n    finally:\n        temporary.unlink(missing_ok=True)\n    return output\n\n\ndef import_site_package(path,root,*,replace=False,allow_demo=False):\n    from forecast_service.pipeline import load_configuration\n    from forecast_service.climate import load_climate\n    root=Path(root)\n    locations,impacts=load_configuration(root)\n    with zipfile.ZipFile(path) as archive:\n        members=archive.infolist()\n        if len(members)>1000 or sum(m.file_size for m in members)>100*1024*1024:\n            raise ValueError(\'Pacchetto troppo grande.\')\n        if len({m.filename for m in members})!=len(members):\n            raise ValueError(\'Nomi duplicati nel pacchetto.\')\n        for member in members:\n            name=member.filename\n            if name not in (\'site.json\',\'climatology.json\') and not re.fullmatch(r\'diagnostics/[A-Za-z0-9_][A-Za-z0-9_.-]*\',name):\n                raise ValueError(\'Percorso non ammesso nel pacchetto.\')\n        payload=json.loads(archive.read(\'site.json\'))\n        slug=validate_payload(payload)\n        if payload[\'data_mode\']==\'demo\' and not allow_demo:\n            raise ValueError(\'Pacchetto dimostrativo: usare --allow-demo soltanto per una prova locale.\')\n        if slug in locations[\'locations\'] and not replace:\n            raise ValueError(\'Località già presente: usare --replace per aggiornarla intenzionalmente.\')\n        updates={}\n        local=copy.deepcopy(payload[\'location\'])\n        if local.get(\'climatology_file\'):\n            raw=archive.read(\'climatology.json\')\n            with tempfile.TemporaryDirectory() as folder:\n                climate_root=Path(folder)\n                (climate_root/\'climatology.json\').write_bytes(raw)\n                load_climate(climate_root,\'climatology.json\')\n            updates[local[\'climatology_file\']]=raw\n        elif \'climatology.json\' in archive.namelist():\n            raise ValueError(\'Climatologia nel pacchetto ma non nella configurazione.\')\n        locations[\'locations\'][slug]=local\n        impacts[\'locations\'][slug]={\'name\':local[\'name\'],\'thresholds\':payload[\'thresholds\']}\n        for name,contents in [(\'locations.json\',locations),(\'impact_thresholds.json\',impacts)]:\n            updates[\'config/\'+name]=(json.dumps(contents,ensure_ascii=False,allow_nan=False,indent=2)+\'\\n\').encode()\n        updates[f\'config/study_sites/{slug}/site.json\']=archive.read(\'site.json\')\n        for member in members:\n            if member.filename.startswith(\'diagnostics/\'):\n                updates[f\'config/study_sites/{slug}/\'+member.filename]=archive.read(member)\n    # Validate merged configuration before touching any existing file.\n    with tempfile.TemporaryDirectory() as folder:\n        check=Path(folder); (check/\'config\').mkdir()\n        for name in (\'locations.json\',\'impact_thresholds.json\'):\n            (check/\'config\'/name).write_bytes(updates[\'config/\'+name])\n        load_configuration(check)\n    old={name:(root/name).read_bytes() if (root/name).exists() else None for name in updates}\n    try:\n        for name,contents in updates.items():\n            destination=root/name; destination.parent.mkdir(parents=True,exist_ok=True)\n            temporary=destination.with_name(destination.name+\'.import.tmp\')\n            temporary.write_bytes(contents); temporary.replace(destination)\n    except Exception:\n        for name,contents in old.items():\n            destination=root/name\n            if contents is None:\n                destination.unlink(missing_ok=True)\n            else:\n                destination.write_bytes(contents)\n        raise\n    return slug\n', 'forecast_service/LICENSE-wave-climate.txt': 'MIT License\n\nCopyright (c) 2026 Alessio Rovere\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n'}
for relative, source in BUNDLED_SOURCES.items():
    target = RUNTIME / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
sys.path.insert(0, str(RUNTIME))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import folium
from IPython.display import display, FileLink
from pyproj import CRS, Transformer
from forecast_service.site_setup import analyze_transects, BEACHFACE_METHOD_VERSION
from forecast_service.era5_download import download_series
from forecast_service.climate import build_climate
from forecast_service.site_package import write_site_package
print("Metodo pendenze:", BEACHFACE_METHOD_VERSION)

In [ ]:
#@title Modalità e salvataggio
DEMO = True #@param {type:"boolean"}
USA_GOOGLE_DRIVE = False #@param {type:"boolean"}
CARTELLA_DRIVE = "OndeARiva_setup" #@param {type:"string"}
if USA_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = Path("/content/drive/MyDrive") / CARTELLA_DRIVE
else:
    BASE = Path("/content/ondeariva_dati") if Path("/content").exists() else RUNTIME / "dati"
BASE.mkdir(parents=True, exist_ok=True)
print("Modalità:", "DIMOSTRATIVA — profili e onde sintetici" if DEMO else "SITO DI STUDIO — dati dell’utente")
print("Cartella:", BASE)
print("La cartella del runtime è temporanea. Per conservare la cache usa Drive oppure scarica il pacchetto finale.")

## 2. Definisci il sito
Le coordinate sono in gradi WGS84. Il punto sulla mappa identifica il sito; quello offshore serve a scaricare onde Copernicus ed ERA5. La cella marina ERA5 verrà verificata durante il download. Il punto del livello marino può essere distinto da quello delle onde.

I valori iniziali sono quelli provvisori di Borghetto; possono essere sostituiti liberamente. Il campo `ID_SITO` deve usare lettere minuscole, numeri e trattini.

In [ ]:
#@title Identità e punti di estrazione
ID_SITO = "borghetto-santo-spirito" #@param {type:"string"}
NOME_SITO = "Borghetto Santo Spirito (SV)" #@param {type:"string"}
TRATTO_DI_COSTA = "Riviera di Ponente · Mar Ligure" #@param {type:"string"}
LAT_SITO = 44.109 #@param {type:"number"}
LON_SITO = 8.242 #@param {type:"number"}
LAT_ONDE = 44.046833 #@param {type:"number"}
LON_ONDE = 8.389593 #@param {type:"number"}
LIVELLO_STESSO_PUNTO = True #@param {type:"boolean"}
LAT_LIVELLO = 44.046833 #@param {type:"number"}
LON_LIVELLO = 8.389593 #@param {type:"number"}
import re
if not re.fullmatch(r"[a-z0-9]+(?:-[a-z0-9]+)*", ID_SITO):
    raise ValueError("ID_SITO non valido.")
for lat, lon in [(LAT_SITO,LON_SITO),(LAT_ONDE,LON_ONDE),(LAT_LIVELLO,LON_LIVELLO)]:
    if not np.isfinite([lat,lon]).all() or not 30<=lat<=46.5 or not -18.2<=lon<=37:
        raise ValueError("Le coordinate devono essere nel dominio mediterraneo.")
WORK = BASE / ID_SITO
WORK.mkdir(parents=True, exist_ok=True)
SITE_INPUT = dict(name=NOME_SITO, area=TRATTO_DI_COSTA,
    map_point=dict(latitude=LAT_SITO,longitude=LON_SITO),
    wave_point=dict(latitude=LAT_ONDE,longitude=LON_ONDE))
MAPPA = folium.Map(location=[LAT_SITO,LON_SITO],zoom_start=12,tiles="OpenStreetMap")
folium.Marker([LAT_SITO,LON_SITO],tooltip="Sito costiero").add_to(MAPPA)
folium.CircleMarker([LAT_ONDE,LON_ONDE],radius=7,color="#006d77",tooltip="Punto onde richiesto").add_to(MAPPA)
folium.LatLngPopup().add_to(MAPPA)
display(MAPPA)
print("Clicca sulla mappa per leggere coordinate, poi riportale nei campi e riesegui questa cella.")

## 3. Carica i profili esportati da QGIS
Usa il CSV del tuo notebook, con colonne predefinite `TR_ID`, `distance`, `SAMPLE_1`, `X`, `Y`. Puoi modificare i nomi sotto. **X e Y devono appartenere a un CRS proiettato con unità in metri**, non essere longitudine e latitudine. `distance` serve a ordinare i punti; la distanza della regressione viene ricalcolata da X/Y come nel notebook fornito.

In modalità demo vengono creati due transetti sintetici, uno con ordine inverso, con un abbassamento del terreno dietro la prima quota sopra `Z_MAX`: quel tratto deve rimanere escluso dalla stima.

In [ ]:
#@title Colonne, coordinate e profili
COL_ID = "TR_ID" #@param {type:"string"}
COL_DIST = "distance" #@param {type:"string"}
COL_Z = "SAMPLE_1" #@param {type:"string"}
COL_X = "X" #@param {type:"string"}
COL_Y = "Y" #@param {type:"string"}
CRS_PROFILI = "EPSG:32632" #@param {type:"string"}
FILE_PROFILI = "" #@param {type:"string"}
SEPARATORE = "," #@param {type:"string"}
DECIMALE = "." #@param {type:"string"}
CORREZIONE_QUOTE_DEM_M = 0.0 #@param {type:"number"}
RIFERIMENTO_QUOTE = "Livello medio mare, da verificare" #@param {type:"string"}
crs = CRS.from_user_input(CRS_PROFILI)
if not crs.is_projected or not all(abs(a.unit_conversion_factor-1)<1e-9 for a in crs.axis_info[:2]):
    raise ValueError("Usa coordinate X/Y in un CRS proiettato in metri.")
if DEMO:
    to_xy = Transformer.from_crs(4326,crs,always_xy=True)
    x0,y0=to_xy.transform(LON_SITO,LAT_SITO)
    demo_rows=[]
    for name,beta,reverse in [("T1",.045,False),("T2",.065,True)]:
        d=np.arange(45)*2.0
        z=beta*d-.3
        z[(d>=40)&(d<=56)]=.4
        xy=np.column_stack([x0+.6*d,y0+.8*d+(10 if name=="T2" else 0)])
        if reverse:
            z=z[::-1]; xy=xy[::-1]
        for i,dist in enumerate(d):
            demo_rows.append({COL_ID:name,COL_DIST:dist,COL_Z:z[i],COL_X:xy[i,0],COL_Y:xy[i,1]})
    profili=pd.DataFrame(demo_rows)
    PROFILE_SOURCE="profili_sintetici"
else:
    if not FILE_PROFILI:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1:
            raise ValueError("Carica un solo CSV di profili.")
        name,content=next(iter(uploaded.items()))
        source=WORK/Path(name).name
        source.write_bytes(content)
    else:
        source=Path(FILE_PROFILI)
    profili=pd.read_csv(source,sep=SEPARATORE,decimal=DECIMALE)
    PROFILE_SOURCE=source.name
if len({COL_ID,COL_DIST,COL_Z,COL_X,COL_Y})!=5:
    raise ValueError("I nomi delle cinque colonne devono essere distinti.")
PROFILE_DATA_MODE="demo" if DEMO else "study"
profili[COL_Z]=pd.to_numeric(profili[COL_Z],errors="raise")+CORREZIONE_QUOTE_DEM_M
print(f"{len(profili)} punti, {profili[COL_ID].nunique()} transetti. Fonte: {PROFILE_SOURCE}")
display(profili.head())

## 4. Calcola la pendenza come nel notebook di riferimento
La procedura:
1. elimina le sole quote mancanti e ordina per `distance`;
2. se la media delle prime 10 quote è maggiore di quella delle ultime 10, inverte il profilo;
3. calcola `x_mare = hypot(X-X_primo, Y-Y_primo)`;
4. si ferma **prima del primo punto con quota maggiore di `Z_MAX`**;
5. fra i punti precedenti usa quelli con quota almeno `Z_MIN` e calcola `np.polyfit(x_mare, quota, 1)[0]`.

La pendenza resta con il proprio segno: un risultato negativo non viene convertito in positivo. Vengono segnalati i profili degeneri o con pendenza fuori da 0–1. Con profili molto corti, le prime/ultime 10 quote possono sovrapporsi: verifica il grafico e, se necessario, scegli esplicitamente l’orientamento.

In [ ]:
#@title Intervallo della battigia e orientamento
Z_MIN = 0.0 #@param {type:"number"}
Z_MAX = 1.0 #@param {type:"number"}
ORIENTAMENTO = "auto" #@param ["auto", "distance_increases_landward", "distance_increases_seaward"]
riepilogo, preparati, selezioni = analyze_transects(profili,Z_MIN,Z_MAX,
    id_col=COL_ID,distance_col=COL_DIST,elevation_col=COL_Z,x_col=COL_X,y_col=COL_Y,orientation=ORIENTAMENTO)
display(riepilogo)
ANALYSIS_SIGNATURE=(Z_MIN,Z_MAX,ORIENTAMENTO,COL_ID,COL_DIST,COL_Z,COL_X,COL_Y,
    hashlib.sha256(pd.util.hash_pandas_object(profili,index=True).values.tobytes()).hexdigest())
riepilogo.to_csv(WORK/"pendenze_transetti.csv",index=False)
validi=riepilogo[riepilogo.status=="valid"]
if validi.empty:
    raise ValueError("Nessun transetto valido: controlla colonne, intervallo e orientamento.")
print("Pendenze valide:",validi.slope.min(),"—",validi.slope.max(),"m/m")
if validi.orientation_ambiguous.any():
    print("Attenzione: orientamento automatico ambiguo per alcuni profili; controlla i grafici.")

In [ ]:
#@title Verifica un transetto nel grafico
ID_TRANSETTO_DA_VEDERE = "" #@param {type:"string"}
identifier=ID_TRANSETTO_DA_VEDERE or str(validi.iloc[0].transect_id)
if identifier not in preparati:
    raise ValueError("Transetto non valido o inesistente; scegli un ID dalla tabella.")
p=preparati[identifier]; sel=selezioni[identifier]
info=validi[validi.transect_id==identifier].iloc[0]
fig,ax=plt.subplots(figsize=(10,4))
ax.plot(p.x_mare,p[COL_Z],color=".35",label="Profilo completo")
ax.scatter(p.loc[sel,"x_mare"],p.loc[sel,COL_Z],color="#d87832",label="Punti usati nella regressione",zorder=3)
xfit=p.loc[sel,"x_mare"]
ax.plot(xfit,info.slope*xfit+info.fit_intercept_m,color="#b54135",ls="--",label="Retta stimata")
ax.axhspan(Z_MIN,Z_MAX,color="#d5e9ee",alpha=.35,label="Intervallo di quota")
if info.first_above_zmax_found:
    ax.axvline(p.loc[int(info.stop_index),"x_mare"],color="#8b633f",ls=":",label="Primo punto sopra Z_MAX")
ax.set(xlabel="Distanza euclidea dal primo punto a mare [m]",ylabel="Quota sul riferimento dichiarato [m]",
       title=f"{'DEMO · ' if DEMO else ''}{identifier}: beta = {info.slope:.4f}, R² = {info.r_squared:.3f}")
ax.legend(fontsize=8); ax.grid(alpha=.2); fig.tight_layout()
fig.savefig(WORK/"profilo_selezionato.png",dpi=170)
PLOT_ANALYSIS_SIGNATURE=ANALYSIS_SIGNATURE
plt.show()

Controlla soprattutto che il primo punto sia dalla parte del mare e che i punti arancioni rappresentino la battigia. L’esclusione dietro il primo superamento di `Z_MAX` riprende il notebook didattico; non costituisce una simulazione della connessione idraulica o dell’inondazione.

In [ ]:
#@title Scegli i transetti e le pendenze da esportare
TRANSETTI_ESCLUSI = "" #@param {type:"string"}
MODALITA_PENDENZE = "Valori dei transetti" #@param ["Valori dei transetti", "Intervallo minimo-massimo"]
NUM_PENDENZE = 5 #@param {type:"integer"}
excluded={v.strip() for v in TRANSETTI_ESCLUSI.split(",") if v.strip()}
unknown=excluded-set(riepilogo.transect_id)
if unknown:
    raise ValueError(f"ID da escludere non riconosciuti: {unknown}")
scelti=riepilogo[~riepilogo.transect_id.isin(excluded)].copy()
if scelti.empty or (scelti.status!="valid").any():
    raise ValueError("Correggi i transetti non validi o escludili esplicitamente prima di esportare.")
if MODALITA_PENDENZE=="Valori dei transetti":
    PENDENZE=sorted(set(float(value) for value in scelti.slope))
else:
    if NUM_PENDENZE<2:
        raise ValueError("Usa almeno due campioni per un intervallo.")
    PENDENZE=sorted(set(float(value) for value in np.linspace(scelti.slope.min(),scelti.slope.max(),NUM_PENDENZE)))
SELECTION_ANALYSIS_SIGNATURE=ANALYSIS_SIGNATURE
print("Pendenze da usare nel portale:",PENDENZE)
print("Gli eventuali valori identici sono deduplicati; gli scenari del portale non pesano i transetti per lunghezza di costa.")

## 5. Stagioni, strutture e riferimento delle quote
Le quote seguenti sono **esempi da definire per il sito**, come nella prima versione del portale. Indica inizio/fine della stagione in formato `MM-GG`; lascia entrambi vuoti se ancora sconosciuti. La correzione del livello marino raccorda il riferimento del modello a quello locale; non va usata per correggere automaticamente il DEM.

In [ ]:
#@title Configurazione delle soglie
INIZIO_ESTATE = "" #@param {type:"string"}
FINE_ESTATE = "" #@param {type:"string"}
QUOTA_OMBRELLONI_M = 0.7 #@param {type:"number"}
QUOTA_STRUTTURE_STAGIONALI_M = 0.9 #@param {type:"number"}
QUOTA_CHIOSCHI_M = 1.1 #@param {type:"number"}
QUOTA_PASSEGGIATA_M = 1.4 #@param {type:"number"}
RUGOSITA_M = 0.001 #@param {type:"number"}
OFFSET_LIVELLO_M = 0.0 #@param {type:"number"}
NOTE_SITO = "Parametri di studio; quote e raccordo verticale da validare." #@param {type:"string"}
import datetime as dt
if bool(INIZIO_ESTATE)!=bool(FINE_ESTATE):
    raise ValueError("Indica entrambe le date estive oppure nessuna.")
for value in (INIZIO_ESTATE,FINE_ESTATE):
    if value: dt.datetime.strptime("2000-"+value,"%Y-%m-%d")
heights=[QUOTA_OMBRELLONI_M,QUOTA_STRUTTURE_STAGIONALI_M,QUOTA_CHIOSCHI_M,QUOTA_PASSEGGIATA_M]
if not np.isfinite(heights).all() or heights!=sorted(set(heights)):
    raise ValueError("Le quattro quote devono essere finite e strettamente crescenti.")
if not np.isfinite([RUGOSITA_M,OFFSET_LIVELLO_M]).all() or RUGOSITA_M<0:
    raise ValueError("Rugosità o correzione del livello non valide.")
thresholds=[]
for identifier,height,label,season in zip(
    ["prima_fila_ombrelloni","strutture_turistiche_stagionali","strutture_turistiche_fisse","infrastrutture_fisse"],heights,
    ["Prima fila di ombrelloni","Strutture turistiche stagionali","Chioschi e strutture fisse","Passeggiata a mare"],
    ["summer","summer","all_year","all_year"]):
    thresholds.append(dict(id=identifier,elevation_m=float(height),short_label=label,description="Possibile interessamento: "+label.lower(),season=season))
display(pd.DataFrame(thresholds)[["short_label","elevation_m","season"]])

## 6. ERA5 — scarica una volta o riusa il tuo CSV
Per dati reali servono un account Climate Data Store e l’accettazione personale delle condizioni di [ERA5 time-series](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=download). La ricerca automatica della cella marina usa anche [ERA5 grigliato](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels?tab=download).

Puoi salvare il token nei **Secrets di Colab**, con nome `CDS_KEY` e accesso a questo notebook, oppure inserirlo nel campo protetto quando richiesto. Non scriverlo nei campi del notebook. La cache contiene soltanto richieste e dati: sarà riutilizzata se la sessione riparte dalla stessa cartella.

Il servizio ERA5 restituisce **periodo medio**, non Tp: lo salviamo come `tm_s`; non lo usiamo come periodo di picco per il runup. Il confronto storico usa Hs. I massimi annuali osservati non sono periodi di ritorno.

In [ ]:
#@title Sorgente e periodo storico
MODALITA_ERA5 = "Scarica ERA5" #@param ["Scarica ERA5", "Importa CSV", "Salta per ora"]
ERA5_INIZIO = "1991-01-01" #@param {type:"date"}
ERA5_FINE = "2020-12-31" #@param {type:"date"}
CONSENSO_DOWNLOAD_ERA5 = False #@param {type:"boolean"}
CELLA_ERA5_NOTA = False #@param {type:"boolean"}
LAT_CELLA_ERA5 = 44.0 #@param {type:"number"}
LON_CELLA_ERA5 = 8.5 #@param {type:"number"}
FILE_CSV_ERA5 = "" #@param {type:"string"}
COL_TEMPO_ERA5 = "time" #@param {type:"string"}
COL_HS_ERA5 = "hs" #@param {type:"string"}
SCARTA_HS_MANCANTI = False #@param {type:"boolean"}
CLIMATE=None
CLIMATE_DATA_MODE="demo" if DEMO else "study"
climate_metadata={}
CACHE=BASE/"_cache_era5"
CACHE.mkdir(parents=True,exist_ok=True)
if DEMO:
    # Synthetic reference, clearly labelled; never presented as an ERA5 download.
    time=pd.date_range("2019-01-01", "2020-12-31 23:00",freq="h",tz="UTC")
    index=np.arange(len(time))
    hs=.25+1.1*(.5+.5*np.sin(index/179))**3+.3*(.5+.5*np.sin(index/37))
    era5_frame=pd.DataFrame({"time":time,"hs_m":hs})
    history=WORK/"clima_dimostrativo.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"Esempio sintetico; non ERA5")
    climate_metadata={"mode":"demo","description":"Distribuzione sintetica per verificare il notebook."}
elif MODALITA_ERA5=="Scarica ERA5":
    if not CONSENSO_DOWNLOAD_ERA5:
        raise ValueError("Accetta le condizioni sul CDS e imposta CONSENSO_DOWNLOAD_ERA5=True per avviare la richiesta.")
    import os
    if not os.environ.get("CDS_KEY"):
        token=None
        try:
            from google.colab import userdata
            token=userdata.get("CDS_KEY")
        except Exception:
            pass
        if not token:
            from getpass import getpass
            token=getpass("Token CDS (input nascosto): ")
        if not token: raise ValueError("Token CDS non disponibile.")
        os.environ["CDS_KEY"]=token
        del token
    known_cell=(LON_CELLA_ERA5,LAT_CELLA_ERA5) if CELLA_ERA5_NOTA else None
    era5_frame,climate_metadata=download_series(SITE_INPUT,dt.date.fromisoformat(ERA5_INIZIO),dt.date.fromisoformat(ERA5_FINE),CACHE,cell=known_cell)
    history=WORK/"ERA5_hourly.csv"; era5_frame.to_csv(history,index=False)
    if climate_metadata["missing_hs_records"] and not SCARTA_HS_MANCANTI:
        raise ValueError("ERA5 ha ore mancanti: CSV conservato. Esamina i dati; puoi autorizzare lo scarto con SCARTA_HS_MANCANTI.")
    if SCARTA_HS_MANCANTI:
        climate_metadata["dropped_missing_rows"]=int(era5_frame.hs_m.isna().sum())
        era5_frame=era5_frame.dropna(subset=["hs_m"])
        history=WORK/"ERA5_hs_valide.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"ERA5")
elif MODALITA_ERA5=="Importa CSV":
    if not FILE_CSV_ERA5:
        from google.colab import files
        uploaded=files.upload()
        if len(uploaded)!=1: raise ValueError("Carica un solo CSV storico.")
        name,content=next(iter(uploaded.items()))
        history=WORK/Path(name).name; history.write_bytes(content)
    else:
        history=Path(FILE_CSV_ERA5)
    era5_frame=pd.read_csv(history)
    if SCARTA_HS_MANCANTI:
        climate_metadata["dropped_missing_rows"]=int(era5_frame[COL_HS_ERA5].isna().sum())
        era5_frame=era5_frame.dropna(subset=[COL_HS_ERA5])
        history=WORK/"ERA5_import_hs_valide.csv"; era5_frame.to_csv(history,index=False)
    CLIMATE=build_climate(history,"ERA5 (CSV dell’utente)",time_column=COL_TEMPO_ERA5,hs_column=COL_HS_ERA5)
else:
    print("Climatologia saltata: la località potrà essere esportata senza riferimento storico.")
if CLIMATE is not None:
    CLIMATE["download"]=climate_metadata
    if not DEMO: CLIMATE["doi"]="10.24381/1cf1ad76"
    (WORK/"climatology.json").write_text(json.dumps(CLIMATE,ensure_ascii=False,allow_nan=False,separators=(",",":")))
    display(pd.DataFrame([{k:CLIMATE[k] for k in ["source","start","end","n_records","p95_m","p99_m","max_hs_m","coverage_fraction"]}]))

In [ ]:
#@title Verifica la distribuzione e i massimi annuali
if CLIMATE is not None:
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    axes[0].hist(CLIMATE["hs_sorted_m"],bins=50,color="#527d78",density=True)
    for key,label,color in [("p95_m","P95","#b39245"),("p99_m","P99","#b65f44")]:
        axes[0].axvline(CLIMATE[key],ls="--",color=color,label=f"{label}: {CLIMATE[key]:.2f} m")
    axes[0].set(xlabel="Hs [m]",ylabel="Densità",title=CLIMATE["source"])
    axes[0].legend()
    annual=pd.DataFrame.from_dict(CLIMATE["annual_maxima"],orient="index")
    axes[1].bar(annual.index,annual.hs_m,color="#527d78")
    axes[1].tick_params(axis="x",labelrotation=60)
    axes[1].set(xlabel="Anno",ylabel="Hs massima osservata [m]",title="Massimi annuali osservati, non valori di ritorno")
    fig.tight_layout(); fig.savefig(WORK/"climatologia.png",dpi=150); plt.show()
    annual.to_csv(WORK/"massimi_annuali.csv",index_label="year")
    display(annual)
    if "cell_lat" in climate_metadata:
        m=folium.Map(location=[LAT_ONDE,LON_ONDE],zoom_start=8)
        folium.Marker([LAT_SITO,LON_SITO],tooltip="Sito").add_to(m)
        folium.CircleMarker([LAT_ONDE,LON_ONDE],radius=6,color="green",tooltip="Punto onde richiesto").add_to(m)
        folium.CircleMarker([climate_metadata["cell_lat"],climate_metadata["cell_lon"]],radius=7,color="orange",tooltip="Cella marina ERA5 usata").add_to(m)
        display(m)
else:
    print("Nessuna climatologia selezionata.")

## 7. Rivedi ed esporta il sito
Il pacchetto contiene configurazione, soglie, pendenze e relativa diagnostica, profili normalizzati e climatologia quando disponibile. **Conserva la cache o il CSV storico in Drive**: il pacchetto riporta il riferimento statistico, non tutti i NetCDF delle richieste.

Un pacchetto prodotto in modalità demo è marcato come tale e l’importatore lo rifiuta per impostazione predefinita. Nessun token entra nel pacchetto. Nessun file del portale viene modificato da Colab.

In [ ]:
#@title Riepilogo prima dell’esportazione
from forecast_service.time_utils import iso
import hashlib
current_signature=(Z_MIN,Z_MAX,ORIENTAMENTO,COL_ID,COL_DIST,COL_Z,COL_X,COL_Y,
    hashlib.sha256(pd.util.hash_pandas_object(profili,index=True).values.tobytes()).hexdigest())
if PROFILE_DATA_MODE!=("demo" if DEMO else "study") or (CLIMATE is not None and CLIMATE_DATA_MODE!=("demo" if DEMO else "study")):
    raise ValueError("Modalità cambiata: riesegui caricamento profili, pendenze e climatologia prima di esportare.")
if current_signature!=ANALYSIS_SIGNATURE or SELECTION_ANALYSIS_SIGNATURE!=ANALYSIS_SIGNATURE or PLOT_ANALYSIS_SIGNATURE!=ANALYSIS_SIGNATURE:
    raise ValueError("Parametri o profili cambiati: riesegui calcolo, grafico e selezione delle pendenze.")
normalized=profili.rename(columns={COL_ID:"TR_ID",COL_DIST:"distance",COL_Z:"SAMPLE_1",COL_X:"X",COL_Y:"Y"})[["TR_ID","distance","SAMPLE_1","X","Y"]]
normalized.to_csv(WORK/"profili_normalizzati.csv",index=False)
location=dict(SITE_INPUT,slopes=PENDENZE,roughness_m=float(RUGOSITA_M),parameters_status="provisional",
    datum_offset_m=float(OFFSET_LIVELLO_M),vertical_reference_status="provisional",
    climatology_file=f"config/climatology/{ID_SITO}.json" if CLIMATE is not None else None,
    summer_season=dict(start_month_day=INIZIO_ESTATE or None,end_month_day=FINE_ESTATE or None,status="configured" if INIZIO_ESTATE else "to_be_defined"),
    transects=json.loads(scelti.to_json(orient="records")),notes=NOTE_SITO)
if not LIVELLO_STESSO_PUNTO:
    location["sealevel_point"]=dict(latitude=LAT_LIVELLO,longitude=LON_LIVELLO)
payload=dict(schema_version=1,slug=ID_SITO,data_mode="demo" if DEMO else "study",location=location,thresholds=thresholds,
    provenance=dict(created_at=iso(dt.datetime.now(dt.timezone.utc)),slope_method=BEACHFACE_METHOD_VERSION,
        reference_notebook="Stockdon_runup_soluzione.ipynb, celle 24–25",profile_source=PROFILE_SOURCE,
        profile_sha256=hashlib.sha256((WORK/"profili_normalizzati.csv").read_bytes()).hexdigest(),
        profile_data_mode=PROFILE_DATA_MODE,climate_data_mode=CLIMATE_DATA_MODE if CLIMATE else None,
        profile_crs=crs.to_string(),vertical_reference=RIFERIMENTO_QUOTE,z_correction_m=CORREZIONE_QUOTE_DEM_M,
        excluded_transects=sorted(excluded),slope_sampling=MODALITA_PENDENZE,z_min_m=Z_MIN,z_max_m=Z_MAX))
display(pd.DataFrame([{"località":NOME_SITO,"modalità":payload["data_mode"],"transetti inclusi":len(scelti),"pendenze":PENDENZE,"ERA5 o riferimento":CLIMATE["source"] if CLIMATE else "Non configurato","stagione estiva":f"{INIZIO_ESTATE} / {FINE_ESTATE}" if INIZIO_ESTATE else "Da definire"}]))
print("Le soglie e il raccordo delle quote restano da validare, anche con profili misurati.")

In [ ]:
#@title Crea e scarica il pacchetto
ESPORTA_PACCHETTO = True #@param {type:"boolean"}
if ESPORTA_PACCHETTO:
    attachments={name:WORK/name for name in ["pendenze_transetti.csv","profili_normalizzati.csv","profilo_selezionato.png"]}
    for name in ["climatologia.png","massimi_annuali.csv"]:
        if CLIMATE is not None and (WORK/name).exists(): attachments[name]=WORK/name
    output=WORK/f"{ID_SITO}_{'DEMO' if DEMO else 'studio'}.zip"
    write_site_package(output,payload,climatology=CLIMATE,attachments=attachments)
    print("Pacchetto creato:",output)
    try:
        from google.colab import files
        files.download(str(output))
    except ImportError:
        display(FileLink(str(output)))

## 8. Importa il pacchetto nella repository
Dopo il download, dal terminale nella cartella `Runup_GUI`, per aggiornare una località già presente:

```sh
python scripts/import_site_package.py /percorso/borghetto-santo-spirito_studio.zip --replace
```

Per una località nuova ometti `--replace`. Il controllo della configurazione viene eseguito prima di scrivere i file. Per una prova locale con un pacchetto demo serve anche `--allow-demo`; non usarlo per pubblicare parametri di esempio come dati reali.

Dopo l’importazione rigenera il bollettino e verifica il risultato nel portale. Il download ERA5 rimane un’attività di setup: i due aggiornamenti giornalieri leggono il riferimento già salvato.

### Riferimenti
- Metodo della pendenza: notebook didattico fornito dall’utente **Stockdon_runup_soluzione.ipynb**, celle 24–25.
- Stockdon, H. F., Holman, R. A., Howd, P. A., & Sallenger, A. H. (2006). *Empirical parameterization of setup, swash, and runup*. Coastal Engineering, 53(7), 573–588. https://doi.org/10.1016/j.coastaleng.2005.12.005
- ERA5 hourly time-series: https://doi.org/10.24381/1cf1ad76. Il downloader adatta lo script fornito dall’utente, copyright 2026 Alessio Rovere, MIT; la licenza è inclusa nella cella tecnica.
- [Colab: caricamento, salvataggio e condivisione dei notebook](https://research.google.com/colaboratory/faq.html).